# Test Interval Method Tutorial

This tutorial estimates cumulative milk yield with the **ICAR Test Interval Method** (TIM).

TIM divides a lactation into three types of contributions:

1. The first test-day yield multiplied by the days from calving to that test.
2. Trapezoids between consecutive test days.
3. The last test-day yield multiplied by the days remaining to the selected horizon.

You will learn how to prepare data, calculate the standard 305-day cumulative milk yield for one lactation, process multiple lactations, and optionally use a shorter lactation length.

First install the lactationcurve package if you have not done this yet with


```bash
% pip install lactationcurve
```

Import the packages

In [2]:
import pandas as pd

from lactationcurve.characteristics.method_test_interval import test_interval_method

c:\Users\Meike van Leerdam\bovi\.venv\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


## 1. Create test-day data

The method accepts common aliases such as `DIM`, `Yield`, and `AnimalId`. We use the canonical names here so the calculation is easy to read.

In [3]:
records = pd.DataFrame(
    {
        "TestId": [101, 101, 101, 101, 101],
        "DaysInMilk": [10, 40, 70, 100, 130],
        "MilkingYield": [20.0, 30.0, 28.0, 26.0, 24.0],
    }
)

records

,TestId,DaysInMilk,MilkingYield
0,101,10,20.0
1,101,40,30.0
2,101,70,28.0
3,101,100,26.0
4,101,130,24.0


## 2. Calculate the 305-day milk yield by hand using the Test Interval Method

For the standard 305-day horizon, the first contribution is `10 × 20`. The interval from DIM 10 to 40 contributes `30 × (20 + 30) / 2`, and so on. The last contribution is `(305 + 1 - 130) × 24` because the implementation counts the endpoint of the calculation window.

In [4]:
ordered = records.sort_values("DaysInMilk").reset_index(drop=True)
ordered["IntervalDays"] = ordered["DaysInMilk"].shift(-1) - ordered["DaysInMilk"]
ordered["AverageYield"] = (ordered["MilkingYield"] + ordered["MilkingYield"].shift(-1)) / 2
ordered["TrapezoidContribution"] = ordered["IntervalDays"] * ordered["AverageYield"]

start_contribution = ordered.loc[0, "DaysInMilk"] * ordered.loc[0, "MilkingYield"]
end_contribution = (305 + 1 - ordered.iloc[-1]["DaysInMilk"]) * ordered.iloc[-1]["MilkingYield"]
intermediate_contribution = ordered["TrapezoidContribution"].sum()

print(f"Start contribution: {start_contribution:.1f} kg")
print(f"Intermediate contribution: {intermediate_contribution:.1f} kg")
print(f"End contribution: {end_contribution:.1f} kg")
print(
    f"Hand-calculated total: "
    f"{start_contribution + intermediate_contribution + end_contribution:.1f} kg"
)
ordered

Start contribution: 200.0 kg
Intermediate contribution: 3180.0 kg
End contribution: 4224.0 kg
Hand-calculated total: 7604.0 kg


,TestId,DaysInMilk,MilkingYield,IntervalDays,AverageYield,TrapezoidContribution
0,101,10,20.0,30.0,25.0,750.0
1,101,40,30.0,30.0,29.0,870.0
2,101,70,28.0,30.0,27.0,810.0
3,101,100,26.0,30.0,25.0,750.0
4,101,130,24.0,NaN,NaN,NaN


## 3. Run the package function

The function performs the same calculation and returns a two-column dataframe. Records beyond `max_dim = 305` are ignored.

In [5]:
result_305 = test_interval_method(records)
result_305

,TestId,LactationMilkYield
0,101,7604.0


## 4. Process multiple lactations

Add another `TestId` and call the same function. Each lactation is sorted and calculated independently.

In [ ]:
# create a second lactation with a different TestId and slightly lower MilkingYield values
second_lactation = records.assign(
    TestId=202,
    MilkingYield=records["MilkingYield"] - 2,
)
all_lactations = pd.concat([records, second_lactation], ignore_index=True)

# call the function with the combined dataframe
test_interval_method(all_lactations)

,TestId,LactationMilkYield
0,101,7604.0
1,202,6992.0


## 5. Use your dataframe without changing column names

Input column names are matched case-insensitively and allow synonymns. This is useful when your source system calls e.g. the columns `DIM` and `Yield` instead of `DaysInMilk` and `MilkingYield`.

In [7]:
# Create a dummy dataframe with the same columns but different names to test
# the aliasing functionality.
alias_records = records.rename(
    columns={"TestId": "AnimalId", "DaysInMilk": "DIM", "MilkingYield": "Yield"}
)

# call the function
test_interval_method(alias_records)

,TestId,LactationMilkYield
0,101,7604.0


## 6. Use a different lactation length

The standard calculation uses 305 days, but you can choose another lactation length with `max_dim`. For example, the following calculates the cumulative milk yield upto day 150. This is useful when you want to compare an early-lactation period or when your data does not cover the full 305 days.

In [8]:
result_150 = test_interval_method(records, max_dim=150)
result_150

,TestId,LactationMilkYield
0,101,3884.0


## 7. Minimum sample requirements

To be able to apply this method 2 or more records are needed. If only one record is available, the function will not calculate a value and give you a message that more measurements are needed.

In [10]:
# Create a dataframe with only one record to test the error handling
records = pd.DataFrame(
    {
        "TestId": [101],
        "DaysInMilk": [10],
        "MilkingYield": [20.0],
    }
)

# Apply the function to the dataframe with only one record. This should raise
# a message that the lactation is skipped.
test_interval_method(records)

Skipping TestId 101: not enough data points for interpolation.


,TestId,LactationMilkYield


## What you learned in this tutorial

- TIM uses actual distances between test days, 
not calendar-month lengths such as with the centering date method.
- At least two test days are needed for a lactation to be included.
- The output keeps the same yield unit as the input (for example, kg or lb).
- The default `max_dim` is 305 days, which is the main result in this tutorial.
- You can set `max_dim=150` to calculate a 150-day yield when that shorter horizon is needed.
- No lactation curves are applied in this method 